## CatBoostClassifier

Данные полученных экспериментов находятся в папке `results`.

In [ ]:
# %pip install catboost

import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve, auc, balanced_accuracy_score, accuracy_score

In [26]:
df_2020 = pd.read_csv(
    r'..\..\preprocessing\data\processed\df_2020_preprocessed.csv')
df_2022 = pd.read_csv(
    r'..\..\preprocessing\data\processed\df_2022_preprocessed.csv')
df_merged = pd.read_csv(r'..\..\preprocessing\data\processed\df_merged.csv')

df_2020.head()

,HeartDisease,BMI,SmokerStatus,AlcoholDrinkers,HadStroke,PhysicalHealthDays,MentalHealthDays,DifficultyWalking,Sex,AgeCategory,RaceEthnicityCategory,HadDiabetes,PhysicalActivities,GeneralHealth,SleepHours,HadAsthma,HadKidneyDisease,HadSkinCancer
0,0,16.60,1,0,0,3.0,30.0,0,Female,55-59,White,Yes,1,Very good,5.0,1,0,1
1,0,20.34,0,0,1,0.0,0.0,0,Female,80 or older,White,No,1,Very good,7.0,0,0,0
2,0,26.58,1,0,0,20.0,30.0,0,Male,65-69,White,Yes,1,Fair,8.0,1,0,0
3,0,24.21,0,0,0,0.0,0.0,0,Female,75-79,White,No,0,Good,6.0,0,0,1
4,0,23.71,0,0,0,28.0,0.0,1,Female,40-44,White,No,1,Very good,8.0,0,0,0


In [27]:
results = pd.DataFrame()

### Обучение на наборе 2020 года

In [28]:
df_2020['AgeCategory'] = df_2020['AgeCategory'].map({
    '18-24': 0,
    '25-29': 1,
    '30-34': 2,
    '35-39': 3,
    '40-44': 4,
    '45-49': 5,
    '50-54': 6,
    '55-59': 7,
    '60-64': 8,
    '65-69': 9,
    '70-74': 10,
    '75-79': 11,
    '80 or older': 12,
})

df_2020['GeneralHealth'] = df_2020['GeneralHealth'].map({
    'Poor': 0,
    'Fair': 1,
    'Good': 2,
    'Very good': 3,
    'Excellent': 4,
})

In [29]:
SEED = 67

X = df_2020.drop('HeartDisease', axis=1)
y = df_2020['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=SEED,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_test,
    y_test,
    train_size=0.5,
    random_state=SEED,
    stratify=y_test
)

In [44]:
model = CatBoostClassifier(
    auto_class_weights='Balanced',
    iterations=1000,
    learning_rate=0.1,
    eval_metric='PRAUC',
    random_seed=SEED,
    verbose=0,
    use_best_model=True
)

model.fit(
    X_train,
    y_train,
    cat_features=['Sex', 'RaceEthnicityCategory', 'HadDiabetes'],
    verbose=100,
    eval_set=(X_valid, y_valid)
)

0:	learn: 0.7994526	test: 0.7969145	best: 0.7969145 (0)	total: 203ms	remaining: 3m 22s
100:	learn: 0.8298407	test: 0.8239178	best: 0.8239385 (98)	total: 21.5s	remaining: 3m 10s
200:	learn: 0.8345941	test: 0.8239255	best: 0.8242009 (147)	total: 44.5s	remaining: 2m 57s
300:	learn: 0.8387048	test: 0.8232318	best: 0.8242009 (147)	total: 1m 8s	remaining: 2m 39s
400:	learn: 0.8420734	test: 0.8225753	best: 0.8242009 (147)	total: 1m 32s	remaining: 2m 18s
500:	learn: 0.8452278	test: 0.8220849	best: 0.8242009 (147)	total: 1m 58s	remaining: 1m 57s
600:	learn: 0.8481650	test: 0.8213114	best: 0.8242009 (147)	total: 2m 24s	remaining: 1m 35s
700:	learn: 0.8510072	test: 0.8206572	best: 0.8242009 (147)	total: 2m 49s	remaining: 1m 12s
800:	learn: 0.8535646	test: 0.8196537	best: 0.8242009 (147)	total: 3m 15s	remaining: 48.5s
900:	learn: 0.8559393	test: 0.8188417	best: 0.8242009 (147)	total: 3m 41s	remaining: 24.3s
999:	learn: 0.8583176	test: 0.8182819	best: 0.8242009 (147)	total: 4m 7s	remaining: 0us

be

CatBoostClassifier(auto_class_weights='Balanced', eval_metric='PRAUC', iterations=1000, learning_rate=0.1, random_seed=67, use_best_model=True, verbose=0)

In [45]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

metrics2020 = {
    'data': '2020',
    'model': 'CatBoostClassifier',
    'roc_auc_score': roc_auc_score(y_test, y_proba),
    'f1_score': f1_score(y_test, y_pred),
    'pr_auc_score': auc(recall, precision),
    'balanced_accuracy_score': balanced_accuracy_score(y_test, y_pred)
}

In [49]:
print(confusion_matrix(y_test, y_pred))

[[21181  7795]
 [  533  2161]]


### Обучение на общем наборе 2020+2022